# Load description for each variable in each pair

Recreates and extends analysis from https://github.com/amit-sharma/chatgpt-causality-pairs
Focuses on analysis of the Tübingen dataset from https://webdav.tuebingen.mpg.de/cause-effect/

In [ ]:
import sys
import os
import time
from dotenv import load_dotenv
from typing import Dict, List, Tuple
import guidance
import os
import pandas as pd
from openai import OpenAI
from portkey_ai import createHeaders
import re
import math
from openai import OpenAI

# Add the notebook setup path to sys.path
notebook_setup_path = os.path.abspath("../")
sys.path.insert(0, notebook_setup_path)

from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()
load_dotenv()


In [ ]:
#azure_model= "o4-mini-2025-04-16"
azure_model="gpt-4.1-2025-04-14"
us_base_url = "https://us.aigw.galileo.roche.com/v1"
portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)
# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
)


In [22]:
from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester
modeler= SimpleModelSuggester(llm=model)

In [23]:
df = pd.read_csv('/home/moleropa/repositories/master/TFM/pywhyllm/docs/notebooks/tuebingen_causality_pairs/tuebingen_pairs.csv')

# Get relationship of each variable pair

In [24]:
llm_output : Dict[str, dict] = {}

In [25]:
# Define parameters for the experiment
temperature = 0.3
num_runs = 1 # Reduced for testing

# Create saved_pairs_info to store ground truth and variable info
saved_pairs_info = {}

# Process only the first 3 rows for testing
#test_df = df.head(1) #or just df all dataset
test_df = df #or just df all dataset
    

In [26]:
def suggest_pairwise_relationship_with_logprobs_flexible(self, variable1: str, variable2: str, openai_client=None, model_name="gpt-4o-mini", log_probs=True, confidence_level=True):
    """
    Suggests a cause-and-effect relationship with optional detailed log probabilities.
    Uses OpenAI client directly for full logprob access when enabled.
    
    Args:
        variable1 (str): The name of the first variable.
        variable2 (str): The name of the second variable.
        openai_client: Optional OpenAI client instance. If None, uses guidance model.
        model_name (str): Model name to use with OpenAI client.
        log_probs (bool): Whether to calculate log probabilities. Default True.
        confidence_level (bool): Whether to request confidence and strength scores. Default True.
        
    Returns:
        dict: Contains 'result', 'description', 'answer', optionally 'logprobs' data, 
              and optionally 'confidence' and 'strength' scores.
    """
    # If no OpenAI client provided, try to create one from environment
    if openai_client is None:
        import os
        # Try to extract connection details from guidance model
        if hasattr(self.llm, 'engine'):
            try:
                openai_client = OpenAI(
                    api_key=os.environ.get("OPENAI_API_KEY"),
                    base_url=os.environ.get("OPENAI_BASE_URL")
                )
            except:
                raise ValueError("Could not create OpenAI client. Please pass openai_client parameter.")
    
    # Confidence and strength instruction
    confidence_instruction = ""
    if confidence_level:
        confidence_instruction = """ 
Additionally, provide TWO scores within tags:
1. <confidence></confidence>: Your confidence in this causal judgment (0-1 scale)
   - How certain are you that you chose the correct relationship (A, B, or C)?
   - Example: High confidence = 0.9 (very sure), Low confidence = 0.5 (uncertain)
   - If the answer is C (no relationship), confidence reflects certainty of no relationship .

2. <strength></strength>: The strength of the causal relationship (0-1 scale)
   - IF a causal relationship exists (A or B), how strong/deterministic is it?
   - 1.0  = Very strong relationship (e.g., "Smoking → Lung Cancer")
   - 0.7 = Moderate relationship (e.g., "Age → Heart Attack" - depends on genetics, lifestyle)
   - 0.5 = Moderate-weak relationship (e.g., "Education → Income" - many exceptions)
   - 0.3 = Weak relationship (e.g., "Birth Order → Personality" - small effect, many confounders)
   - 0.0 = No causal relationship (if the answer is C)
   
Important distinctions:
- You can have HIGH confidence (0.9) that a WEAK relationship (0.3) exists
- You can have LOW confidence (0.5) about a potentially STRONG relationship (0.8)
- If the answer is C (no relationship), set strength to 0.0 but confidence reflects certainty of no relationship and confidence can be 1.0 (very sure no relationship exists) 
- Strength reflects the SIZE of the causal effect, not your confidence in the judgment
"""
    
    # Construct the prompt
    prompt = f"""Which cause-and-effect-relationship is more likely? Provide reasoning and give your final answer (A, B, or C) in <answer> </answer> tags with the letter only and no whitespaces.{confidence_instruction}
A. {variable1} causes {variable2} 
B. {variable2} causes {variable1} 
C. neither {variable1} nor {variable2} cause each other."""
    
    # Make API call with conditional logprobs
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a helpful assistant for causal reasoning."},
            {"role": "user", "content": prompt}
        ],
     #   temperature=0.0,
        logprobs=log_probs,  # Use the boolean parameter
        top_logprobs=3 if log_probs else None,  # Only request top logprobs if needed
        #max_tokens=300  # Increased to accommodate strength explanation
    )
    
    choice = response.choices[0]
    description = choice.message.content
    
    # Extract answer
    answer = re.findall(r"<answer>(.*?)</answer>", description)
    answer = [ans.strip() for ans in answer]
    answer_str = "".join(answer) if answer else ""

    # Extract confidence and strength if requested
    confidence_score = None
    strength_score = None
    
    if confidence_level:
        # Extract confidence
        confidence_match = re.findall(r"<confidence>(.*?)</confidence>", description)
        if confidence_match:
            try:
                confidence_score = float(confidence_match[0].strip())
                confidence_score = max(0.0, min(1.0, confidence_score))  # Clamp to [0, 1]
            except ValueError:
                confidence_score = None
        
        # Extract strength
        strength_match = re.findall(r"<strength>(.*?)</strength>", description)
        if strength_match:
            try:
                strength_score = float(strength_match[0].strip())
                strength_score = max(0.0, min(1.0, strength_score))  # Clamp to [0, 1]
            except ValueError:
                strength_score = None

    # Determine result based on the chosen answer
    if answer_str == "A":
        result = [variable1, variable2, description]
    elif answer_str == "B":
        result = [variable2, variable1, description]
    elif answer_str == "C":
        result = [None, None, description]
    else:
        result = [None, None, description]

    # Base return dictionary
    return_dict = {
        'result': result,
        'description': description,
        'answer': answer_str,
    }
    
    # Add confidence and strength scores if requested
    if confidence_level:
        return_dict['confidence_score'] = confidence_score
        return_dict['strength_score'] = strength_score
    
    # Only process logprobs if requested
    if log_probs:
        logprobs_data = []
        if getattr(choice, "logprobs", None) and getattr(choice.logprobs, "content", None):
            logprobs_data = choice.logprobs.content

        answer_token_logprob = None
        answer_choice_logprobs = {}
        answer_tokens_logprobs = []

        if logprobs_data:
            reconstructed_text = ""
            token_spans = []
            for token_item in logprobs_data:
                token_text = getattr(token_item, "token", "") or ""
                start_idx = len(reconstructed_text)
                reconstructed_text += token_text
                end_idx = len(reconstructed_text)
                token_spans.append((start_idx, end_idx, token_item))

            match = re.search(r"<answer>(.*?)</answer>", reconstructed_text, flags=re.DOTALL)
            if match:
                content_start, content_end = match.span(1)
                for start_idx, end_idx, token_item in token_spans:
                    overlaps_answer = start_idx < content_end and end_idx > content_start
                    if overlaps_answer:
                        token_text = getattr(token_item, "token", "") or ""
                        token_logprob = getattr(token_item, "logprob", None)
                        answer_tokens_logprobs.append({
                            "token": token_text,
                            "logprob": token_logprob
                        })

                        # FIXED: Clean token properly to handle ">A", "<answer", etc.
                        token_clean = ''.join(c for c in token_text if c.isalnum())
                        if token_clean in {"A", "B", "C"} and token_logprob is not None:
                            answer_choice_logprobs[token_clean] = token_logprob

                        top_alternatives = getattr(token_item, "top_logprobs", None) or []
                        for alt in top_alternatives:
                            alt_token = getattr(alt, "token", "") or ""
                            alt_clean = ''.join(c for c in alt_token if c.isalnum())
                            if alt_clean in {"A", "B", "C"}:
                                alt_logprob = getattr(alt, "logprob", None)
                                if alt_logprob is not None:
                                    answer_choice_logprobs[alt_clean] = alt_logprob
                    
                    if start_idx <= content_start and end_idx > content_start:
                        if answer_token_logprob is None:
                            answer_token_logprob = getattr(token_item, "logprob", None)

        # Add logprobs data to return dictionary
        return_dict.update({
            'logprobs': logprobs_data,
            'answer_token_logprob': answer_token_logprob,
            'answer_choice_logprobs': answer_choice_logprobs,
            'answer_tokens_logprobs': answer_tokens_logprobs,
        })

    return return_dict

In [27]:
from types import MethodType
modeler.suggest_pairwise_relationship_with_logprobs_flexible = MethodType(
    suggest_pairwise_relationship_with_logprobs_flexible, modeler
)

In [29]:
# Iterate through each pair and run multiple times
start = time.time()

for pair_number, values in test_df.iterrows():
    pair_id = f"pair{pair_number:04d}"  # Create pair ID like "pair0001", "pair0002", etc.

    saved_pairs_info[pair_id] = {
        "var1": values['var1'],
        "var2": values['var2'],
        "ground_truth": values['ground_truth'],  # columna en tu dataframe
    }
    
    for n in range(1, num_runs + 1):  # Run 1 to 5
        temp_dict = {}
        
        print(f"Processing {pair_id}, run {n}/{num_runs}")
        
        # Test A -> B direction
        temp_dict['llm_ab'] = modeler.suggest_pairwise_relationship_with_logprobs_flexible(
            variable1=values['var1'], 
            variable2=values['var2'], 
            openai_client=azure_openai_client,
            model_name=azure_model,
            confidence_level=True,  # Set to True to enable confidence scoring
            log_probs=False,
        )

        # Test B -> A direction
        temp_dict['llm_ba'] = modeler.suggest_pairwise_relationship_with_logprobs_flexible(
            variable1=values['var2'],
            variable2=values['var1'],
            openai_client=azure_openai_client,
            model_name=azure_model,
            confidence_level=True,  # Set to True to enable confidence scoring
            log_probs=False,
        )
        
        # Store results with key: (pair_id, temperature, run_number)
        llm_output[(pair_id, temperature, n )] = temp_dict
        
        print(f"  A->B: {temp_dict['llm_ab']}, B->A: {temp_dict['llm_ba']}")

# Calculate latencies after all processing is complete
total_time = time.time() - start
print(f"\nCompleted processing {len(test_df)} pairs with {num_runs} runs each")
print(f"Total execution time: {total_time:.2f}s")

# Calculate latencies
avg_latency_per_pair = total_time / (len(test_df) * num_runs)  # Time per pair (both A->B and B->A) per run
avg_latency_per_run = total_time / (len(test_df) * num_runs * 2)  # Time per individual query (A->B or B->A)

print(f"Average time per pair (A->B + B->A): {avg_latency_per_pair:.2f}s")
print(f"Average time per individual query: {avg_latency_per_run:.2f}s")

Processing pair0000, run 1/1
  A->B: {'result': [' Altitude', ' Temperature', 'Altitude refers to the height above sea level. As altitude increases, the atmosphere becomes thinner and holds less heat, which generally leads to lower temperatures. So, altitude is a physical and geographic factor that affects temperature.\n\nOn the other hand, temperature does not cause altitude—variation in temperature does not affect how high a location is above sea level.\n\nTherefore, the most likely causal relationship is that altitude causes temperature.\n\n<answer>A</answer>\n<confidence>0.95</confidence>\n<strength>0.8</strength>'], 'description': 'Altitude refers to the height above sea level. As altitude increases, the atmosphere becomes thinner and holds less heat, which generally leads to lower temperatures. So, altitude is a physical and geographic factor that affects temperature.\n\nOn the other hand, temperature does not cause altitude—variation in temperature does not affect how high a loc

### Export results

In [30]:
results : Dict = {}

for id in saved_pairs_info:

    av_correct_ab = 0
    av_correct_ba = 0
    
    # Lists to collect confidence, strength scores, and references across runs
    confidence_scores_ab = []
    confidence_scores_ba = []
    strength_scores_ab = []
    strength_scores_ba = []
    references_ab = []
    references_ba = []

    for i in range(num_runs):
        # Extract relationship, confidence, strength, and other data from dictionary
        ab_result = llm_output[(id, 0.3, i+1)]['llm_ab']
        ba_result = llm_output[(id, 0.3, i+1)]['llm_ba']
        
        # Get data from result dictionary
        ab_answer = ab_result.get('answer', '')
        ba_answer = ba_result.get('answer', '')
        ab_confidence = ab_result.get('confidence_score')
        ba_confidence = ba_result.get('confidence_score')
        ab_strength = ab_result.get('strength_score')
        ba_strength = ba_result.get('strength_score')
        
        # Store confidence scores (only if not None)
        if ab_confidence is not None:
            confidence_scores_ab.append(ab_confidence)
        if ba_confidence is not None:
            confidence_scores_ba.append(ba_confidence)
        
        # Store strength scores (only if not None)
        if ab_strength is not None:
            strength_scores_ab.append(ab_strength)
        if ba_strength is not None:
            strength_scores_ba.append(ba_strength)
            
        # For baseline method, we don't have references, so use empty lists
        references_ab.extend([])
        references_ba.extend([])

        # Convert answer letters to relationship values for correctness checking
        # A means var1 -> var2 (relationship = 1), B means var2 -> var1 (relationship = 0), C means no relationship
        ab_relationship = 1 if ab_answer == "A" else 0 if ab_answer == "B" else None
        ba_relationship = 1 if ba_answer == "A" else 0 if ba_answer == "B" else None

        # Check correctness using the relationship value
        if ab_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ab += 1
        elif ab_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ab += 1

        if ba_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " L":
            av_correct_ba += 1
        elif ba_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " R":
            av_correct_ba += 1

    av_correct_ab /= num_runs
    av_correct_ba /= num_runs
    
    # Calculate average confidence scores
    avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
    avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None
    
    # Calculate average strength scores
    avg_strength_ab = sum(strength_scores_ab) / len(strength_scores_ab) if strength_scores_ab else None
    avg_strength_ba = sum(strength_scores_ba) / len(strength_scores_ba) if strength_scores_ba else None

    temp : Dict = {}

    temp['PairID'] = id
    temp['CorrectACauseB'] = av_correct_ab
    temp['CorrectBCauseA'] = av_correct_ba
    temp['VarA'] = saved_pairs_info[id]['var1']
    temp['VarB'] = saved_pairs_info[id]['var2']
    temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']
    temp['ConfidenceAB'] = avg_confidence_ab
    temp['ConfidenceBA'] = avg_confidence_ba
    temp['StrengthAB'] = avg_strength_ab  # NEW: Average strength score for A->B
    temp['StrengthBA'] = avg_strength_ba  # NEW: Average strength score for B->A
    temp['ReferencesAB'] = references_ab
    temp['ReferencesBA'] = references_ba

    results[id] = temp
    print(results[id])

#with logprobs
# results : Dict = {}

# for id in saved_pairs_info:

#     av_correct_ab = 0
#     av_correct_ba = 0
    
#     # Lists to collect confidence scores, logprobs, and references across runs
#     confidence_scores_ab = []
#     confidence_scores_ba = []
#     answer_token_logprobs_ab = []
#     answer_token_logprobs_ba = []
#     ab_answer_tokens_logprobs = []
#     ba_answer_tokens_logprobs = []
#     references_ab = []
#     references_ba = []

#     for i in range(num_runs):
#         # Extract relationship, confidence, and other data from dictionary
#         ab_result = llm_output[(id, 0.3, i+1)]['llm_ab']
#         ba_result = llm_output[(id, 0.3, i+1)]['llm_ba']
        
#         # Get data from result dictionary
#         ab_answer = ab_result.get('answer', '')
#         ba_answer = ba_result.get('answer', '')
#         ab_confidence = ab_result.get('confidence_score')
#         ba_confidence = ba_result.get('confidence_score')
        
#         # Get logprobs data
#         ab_token_logprob = ab_result.get('answer_token_logprob')
#         ba_token_logprob = ba_result.get('answer_token_logprob')
#         ab_answer_tokens_logprobs = ab_result.get('answer_tokens_logprobs', {})
#         ba_answer_tokens_logprobs = ba_result.get('answer_tokens_logprobs', {})
        
#         # Store confidence scores (only if not None)
#         if ab_confidence is not None:
#             confidence_scores_ab.append(ab_confidence)
#         if ba_confidence is not None:
#             confidence_scores_ba.append(ba_confidence)
        
#         # Store logprobs (only if not None)
#         if ab_token_logprob is not None:
#             answer_token_logprobs_ab.append(ab_token_logprob)
#         if ba_token_logprob is not None:
#             answer_token_logprobs_ba.append(ba_token_logprob)
        
#         # Store choice logprobs (for all A/B/C options)
#         if ab_answer_tokens_logprobs:
#             ab_answer_tokens_logprobs.append(ab_answer_tokens_logprobs)
#         if ba_answer_tokens_logprobs:
#             ba_answer_tokens_logprobs.append(ba_answer_tokens_logprobs)

#         # For baseline method, we don't have references, so use empty lists
#         references_ab.extend([])
#         references_ba.extend([])

#         # Convert answer letters to relationship values for correctness checking
#         # A means var1 -> var2 (relationship = 1), B means var2 -> var1 (relationship = 0), C means no relationship
#         ab_relationship = 1 if ab_answer == "A" else 0 if ab_answer == "B" else None
#         ba_relationship = 1 if ba_answer == "A" else 0 if ba_answer == "B" else None

#         # Check correctness using the relationship value
#         if ab_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ab += 1
#         elif ab_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ab += 1

#         if ba_relationship == 1 and saved_pairs_info[id]['ground_truth'] == " L":
#             av_correct_ba += 1
#         elif ba_relationship == 0 and saved_pairs_info[id]['ground_truth'] == " R":
#             av_correct_ba += 1

#     av_correct_ab /= num_runs
#     av_correct_ba /= num_runs
    
#     # Calculate average confidence scores
#     avg_confidence_ab = sum(confidence_scores_ab) / len(confidence_scores_ab) if confidence_scores_ab else None
#     avg_confidence_ba = sum(confidence_scores_ba) / len(confidence_scores_ba) if confidence_scores_ba else None
    
#     # Calculate average answer token logprobs
#     avg_answer_token_logprob_ab = sum(answer_token_logprobs_ab) / len(answer_token_logprobs_ab) if answer_token_logprobs_ab else None
#     avg_answer_token_logprob_ba = sum(answer_token_logprobs_ba) / len(answer_token_logprobs_ba) if answer_token_logprobs_ba else None

#     # Format answer_tokens_logprobs as strings for storage
#     tokens_logprobs_ab_str = "; ".join([str(d) for d in ab_answer_tokens_logprobs]) if ab_answer_tokens_logprobs else ""
#     tokens_logprobs_ba_str = "; ".join([str(d) for d in ba_answer_tokens_logprobs]) if ba_answer_tokens_logprobs else ""

#     temp : Dict = {}

#     temp['PairID'] = id
#     temp['CorrectACauseB'] = av_correct_ab
#     temp['CorrectBCauseA'] = av_correct_ba
#     temp['VarA'] = saved_pairs_info[id]['var1']
#     temp['VarB'] = saved_pairs_info[id]['var2']
#     temp['GroundTruth'] = saved_pairs_info[id]['ground_truth']
#     temp['ConfidenceAB'] = avg_confidence_ab
#     temp['ConfidenceBA'] = avg_confidence_ba
#     temp['AnswerTokenLogprobAB'] = avg_answer_token_logprob_ab  # Average logprob of the answer token
#     temp['AnswerTokenLogprobBA'] = avg_answer_token_logprob_ba  # Average logprob of the answer token
#     temp['AnswerTokensLogprobsAB'] = tokens_logprobs_ab_str  # All A/B/C logprobs across runs
#     temp['AnswerTokensLogprobsBA'] = tokens_logprobs_ba_str  # All A/B/C logprobs across runs
#     temp['ReferencesAB'] = references_ab
#     temp['ReferencesBA'] = references_ba

#     results[id] = temp
#     print(results[id])

{'PairID': 'pair0000', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Temperature', 'GroundTruth': ' R', 'ConfidenceAB': 0.95, 'ConfidenceBA': 1.0, 'StrengthAB': 0.8, 'StrengthBA': 0.8, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0001', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Precipitation', 'GroundTruth': ' R', 'ConfidenceAB': 0.95, 'ConfidenceBA': 0.95, 'StrengthAB': 0.7, 'StrengthBA': 0.7, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0002', 'CorrectACauseB': 0.0, 'CorrectBCauseA': 0.0, 'VarA': ' Longitude', 'VarB': ' Temperature', 'GroundTruth': ' R', 'ConfidenceAB': 1.0, 'ConfidenceBA': 1.0, 'StrengthAB': 0.0, 'StrengthBA': 0.0, 'ReferencesAB': [], 'ReferencesBA': []}
{'PairID': 'pair0003', 'CorrectACauseB': 1.0, 'CorrectBCauseA': 1.0, 'VarA': ' Altitude', 'VarB': ' Sunshine hours', 'GroundTruth': ' R', 'ConfidenceAB': 0.8, 'ConfidenceBA': 0.85, 'StrengthAB': 0.3, 'StrengthBA': 0.3, 'Reference

In [31]:
# Calculate accuracy metrics
accuracy_results = {}

total_pairs = len(results)
sum_ab_accuracy = 0
sum_ba_accuracy = 0
sum_joint_accuracy = 0
sum_ab_confidence = 0
sum_ba_confidence = 0
sum_ab_strength = 0
sum_ba_strength = 0
count_ab_confidence = 0
count_ba_confidence = 0
count_ab_strength = 0
count_ba_strength = 0


for pair_id, result in results.items():
    # Individual accuracies per pair (these are already averages from multiple runs, can be decimal)
    correct_ab = result['CorrectACauseB']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    correct_ba = result['CorrectBCauseA']  # This is the mean accuracy (0.0 to 1.0, can be decimal)
    
    # Get confidence scores
    confidence_ab = result.get('ConfidenceAB')
    confidence_ba = result.get('ConfidenceBA')
    
    # Get strength scores
    strength_ab = result.get('StrengthAB')
    strength_ba = result.get('StrengthBA')
    
    # Joint accuracy: average of both directions (more nuanced approach)
    joint_accuracy = (correct_ab + correct_ba) / 2.0
    
    # Sum for overall statistics (averaging across all pairs)
    sum_ab_accuracy += correct_ab
    sum_ba_accuracy += correct_ba
    sum_joint_accuracy += joint_accuracy
    
    # Sum confidence scores for overall statistics
    if confidence_ab is not None:
        sum_ab_confidence += confidence_ab
        count_ab_confidence += 1
    if confidence_ba is not None:
        sum_ba_confidence += confidence_ba
        count_ba_confidence += 1
    
    # Sum strength scores for overall statistics
    if strength_ab is not None:
        sum_ab_strength += strength_ab
        count_ab_strength += 1
    if strength_ba is not None:
        sum_ba_strength += strength_ba
        count_ba_strength += 1
    
    # STORING INDIVIDUAL
    # Store individual pair results (only decimal values, no redundant percentages)
    accuracy_results[pair_id] = {
        'PairID': pair_id,
        'VarA': result['VarA'],
        'VarB': result['VarB'],
        'GroundTruth': result['GroundTruth'],
        'AccuracyAB': correct_ab,  # Decimal value (0.0 to 1.0)
        'AccuracyBA': correct_ba,  # Decimal value (0.0 to 1.0)
        'JointAccuracy': joint_accuracy,  # Average of both directions
        'ConfidenceAB': confidence_ab,  # Confidence score for A->B
        'ConfidenceBA': confidence_ba,  # Confidence score for B->A
        'StrengthAB': strength_ab,  # Strength score for A->B
        'StrengthBA': strength_ba   # Strength score for B->A
    }
    
    print(f"Pair {pair_id}: {result['VarA']} -> {result['VarB']}")
    print(f"  Ground Truth: {result['GroundTruth']}")
    print(f"  A→B Accuracy: {correct_ab:.3f}, Confidence: {f'{confidence_ab:.3f}' if confidence_ab is not None else 'N/A'}, Strength: {f'{strength_ab:.3f}' if strength_ab is not None else 'N/A'}")
    print(f"  B→A Accuracy: {correct_ba:.3f}, Confidence: {f'{confidence_ba:.3f}' if confidence_ba is not None else 'N/A'}, Strength: {f'{strength_ba:.3f}' if strength_ba is not None else 'N/A'}")
    print(f"  Joint Accuracy (avg): {joint_accuracy:.3f}")
    print()


# Overall accuracy statistics (averages across all pairs)
overall_ab_accuracy = sum_ab_accuracy / total_pairs
overall_ba_accuracy = sum_ba_accuracy / total_pairs
overall_joint_accuracy = sum_joint_accuracy / total_pairs

# Overall confidence statistics
overall_ab_confidence = sum_ab_confidence / count_ab_confidence if count_ab_confidence > 0 else None
overall_ba_confidence = sum_ba_confidence / count_ba_confidence if count_ba_confidence > 0 else None

# Overall strength statistics
overall_ab_strength = sum_ab_strength / count_ab_strength if count_ab_strength > 0 else None
overall_ba_strength = sum_ba_strength / count_ba_strength if count_ba_strength > 0 else None

print("=== OVERALL ACCURACY STATISTICS ===")
print(f"Total pairs processed: {total_pairs}")
print(f"\nMEAN ACCURACIES (across all pairs):")
print(f"A→B Mean Accuracy: {overall_ab_accuracy:.3f}")
print(f"B→A Mean Accuracy: {overall_ba_accuracy:.3f}")
print(f"Joint Mean Accuracy: {overall_joint_accuracy:.3f}")
print(f"\nMEAN CONFIDENCE SCORES:")
print(f"A→B Mean Confidence: {f'{overall_ab_confidence:.3f}' if overall_ab_confidence is not None else 'N/A'}")
print(f"B→A Mean Confidence: {f'{overall_ba_confidence:.3f}' if overall_ba_confidence is not None else 'N/A'}")
print(f"\nLATENCY METRICS:")
print(f"Average time per pair (both directions): {avg_latency_per_pair:.2f}s")
print(f"Average time per query: {avg_latency_per_run:.2f}s")

# Summary statistics (only valuable info, no redundant percentages)
accuracy_summary = {
    'total_pairs': total_pairs,
    'ab_mean_accuracy': overall_ab_accuracy,
    'ba_mean_accuracy': overall_ba_accuracy,
    'joint_mean_accuracy': overall_joint_accuracy,
    'ab_mean_confidence': overall_ab_confidence,
    'ba_mean_confidence': overall_ba_confidence,
    'avg_latency_per_pair_seconds': round(avg_latency_per_pair, 2),
    'avg_latency_per_query_seconds': round(avg_latency_per_run, 2),
    'total_execution_time_seconds': round(total_time, 2)
}

Pair pair0000:  Altitude ->  Temperature
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.950, Strength: 0.800
  B→A Accuracy: 1.000, Confidence: 1.000, Strength: 0.800
  Joint Accuracy (avg): 1.000

Pair pair0001:  Altitude ->  Precipitation
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.950, Strength: 0.700
  B→A Accuracy: 1.000, Confidence: 0.950, Strength: 0.700
  Joint Accuracy (avg): 1.000

Pair pair0002:  Longitude ->  Temperature
  Ground Truth:  R
  A→B Accuracy: 0.000, Confidence: 1.000, Strength: 0.000
  B→A Accuracy: 0.000, Confidence: 1.000, Strength: 0.000
  Joint Accuracy (avg): 0.000

Pair pair0003:  Altitude ->  Sunshine hours
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.800, Strength: 0.300
  B→A Accuracy: 1.000, Confidence: 0.850, Strength: 0.300
  Joint Accuracy (avg): 1.000

Pair pair0004:  Age ->  Length
  Ground Truth:  R
  A→B Accuracy: 1.000, Confidence: 0.950, Strength: 0.700
  B→A Accuracy: 1.000, Confidence: 0.950, Strength: 0.700


In [32]:
# Save accuracy results to CSV
import csv

# CSV file for detailed accuracy results (now includes confidence and strength scores)
accuracy_csv_file = "alba_accuracy_results_m1_4.1.csv"

# Define headers for detailed accuracy results (includes confidence and strength)
accuracy_header = [
    "PairID", "VarA", "VarB", "GroundTruth", 
    "AccuracyAB", "AccuracyBA", "JointAccuracy",
    "ConfidenceAB", "ConfidenceBA",
    "StrengthAB", "StrengthBA"
]

# Write detailed accuracy results
with open(accuracy_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=accuracy_header)
    writer.writeheader()
    for pair_id, values in accuracy_results.items():
        writer.writerow(values)

print(f"Detailed accuracy CSV file '{accuracy_csv_file}' has been created.")

# CSV file for summary statistics
summary_csv_file = "alba_accuracy_summary_m1_4.1.csv"

# Write summary statistics
with open(summary_csv_file, mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(accuracy_summary.keys()))
    writer.writeheader()
    writer.writerow(accuracy_summary)

print(f"Summary accuracy CSV file '{summary_csv_file}' has been created.")

ab_conf_str = f"{accuracy_summary['ab_mean_confidence']:.3f}" if accuracy_summary['ab_mean_confidence'] is not None else 'N/A'
ba_conf_str = f"{accuracy_summary['ba_mean_confidence']:.3f}" if accuracy_summary['ba_mean_confidence'] is not None else 'N/A'


Detailed accuracy CSV file 'alba_accuracy_results_m1_4.1.csv' has been created.
Summary accuracy CSV file 'alba_accuracy_summary_m1_4.1.csv' has been created.
